## Word Level LSTM

### Imports

In [1]:
from keras.models import Sequential
from keras.layers import Activation,LSTM,Dense
from tensorflow.keras.optimizers import Adam

In [2]:
import pandas as pd
import numpy as np
import re

### Helper functions

In [3]:
def split_str(delimiters, string, maxsplit=0):
    regex_pattern = '|'.join(map(re.escape, delimiters))
    return re.split(regex_pattern, string, maxsplit)

### Read data

In [4]:
rap_df = pd.read_csv("./../dataset/preprocessed_rap.csv")
rap_df = rap_df.drop('Unnamed: 0', axis=1)
rap_df = rap_df.drop('Unnamed: 0.1', axis=1)

In [5]:
rap_df.shape

(10077, 5)

In [6]:
rap_df.head(5)

,artist,genre,title,lyrics,word_num
0,Snoop Dogg,rap,Gin and Juice,"(Ugh) Ha-ha-ha, I'm serious, nigga One of y'al...",618
1,Snoop Dogg,rap,Drop It Like It’s Hot,"Snoop Snoop When the pimp's in the crib, ma D...",781
2,Snoop Dogg,rap,Ain’t No Fun (If the Homies Can’t Have None),You're back now at the jack-off hour This is D...,599
3,Snoop Dogg,rap,Murder Was the Case (Death After Visualizing E...,"(""Indo Smoke"" Plays in Background) Aye, aye, ...",631
4,Snoop Dogg,rap,Who Am I (What’s My Name)?,EeeyiyiyiyiyahtheDoggPound'sinthehou-owwse (th...,428


In [76]:
train_sz = 1000
train_rap = rap_df.iloc[:train_sz]
print(train_rap.shape[0])

1000


### Training corpus

In [77]:
words_corpus=''
for index,row in train_rap.iterrows():
    words_corpus += " "
    words_corpus += row["title"]
    lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  row["lyrics"].lower().strip())
    lyric = [s for s in lyric if s!=''][0:100]
    words_corpus += " " + " ".join(lyric)
cleaned_corpus = words_corpus.split(" ")
cleaned_corpus.extend([";", ",", "-", "\\", "'", '"', "(", ")", "[", "]",  ".", "?", "!", ":", "{", "}"])


In [78]:
vocab = list(set(cleaned_corpus))
print(len(vocab))
# print(vocab)
word_ix={c:i for i,c in enumerate(vocab)}
ix_word={i:c for i,c in enumerate(vocab)}

9182


In [21]:
# word_ix

In [20]:
# ix_word

In [21]:
import json
with open("./../dataset/word_ix.json", "w") as write_file:
    json.dump(word_ix, write_file, indent=4)
with open("./../dataset/ix_word.json", "w") as write_file:
    json.dump(ix_word, write_file, indent=4)

### Word2vec embeddings

In [79]:
rap_sentences = []
for index, row in train_rap.iterrows():
    lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  row["lyrics"].lower().strip())
    lyric = [s for s in lyric if s!=''][0:100]
    rap_sentences.append(lyric)

In [80]:
print(len(rap_sentences))
print(rap_sentences[0])

1000
['ugh', 'ha', 'ha', 'ha', 'i', 'm', 'serious', 'nigga', 'one', 'of', 'y', 'all', 'niggas', 'got', 'some', 'bad', 'motherfuckin', 'breath', 'oh', 'man', 'aye', 'baby', 'aye', 'baby', 'shit', 'aye', 'baby', 'get', 'some', 'bubblegum', 'in', 'this', 'motherfucker', 'or', 'somethin', 'aye', 'nigga', 'get', 'somethin', 'to', 'eat', 'dog', 'aye', 'nigga', 'study', 'long', 'study', 'wrong', 'nigga', 'with', 'so', 'much', 'drama', 'in', 'the', 'l', 'b', 'c', 'it', 's', 'kind', 'of', 'hard', 'bein', 'snoop', 'd', 'o', 'double', 'g', 'but', 'i', 'somehow', 'some', 'way', 'keep', 'comin', 'up', 'with', 'funky', 'ass', 'shit', 'like', 'every', 'single', 'day', 'may', 'i', 'kick', 'a', 'little', 'something', 'for', 'the', 'g', 's', 'and', 'make', 'a', 'few', 'ends']


In [81]:
from gensim.models import Word2Vec

In [82]:
word_model = Word2Vec(sentences=rap_sentences, vector_size=300, window=5, min_count=1, workers=4)
word_model.save("./../dataset/word2vec_300.model")

In [83]:
word_model = Word2Vec.load("./../dataset/word2vec_300.model")

In [84]:
embed = word_model.wv['a']
print(embed.shape)

(300,)


### Training data

In [85]:
vec_size = 300
num_words = 10

In [86]:
sentences=[]
next_word=[]
for index, row in train_rap.iterrows():
    lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  row["lyrics"].lower().strip())
    lyric = [s for s in lyric if s!=''][0:100]

    for i in range(len(lyric)-num_words-1):        
        sentences.append(" ".join(lyric[i:i+num_words]))
        next_word.append(lyric[i+num_words])

In [87]:
print(len(sentences))
sentences[0:10]

87784


['ugh ha ha ha i m serious nigga one of',
 'ha ha ha i m serious nigga one of y',
 'ha ha i m serious nigga one of y all',
 'ha i m serious nigga one of y all niggas',
 'i m serious nigga one of y all niggas got',
 'm serious nigga one of y all niggas got some',
 'serious nigga one of y all niggas got some bad',
 'nigga one of y all niggas got some bad motherfuckin',
 'one of y all niggas got some bad motherfuckin breath',
 'of y all niggas got some bad motherfuckin breath oh']

In [88]:
print(len(next_word))
print(next_word[0:10])

87784
['y', 'all', 'niggas', 'got', 'some', 'bad', 'motherfuckin', 'breath', 'oh', 'man']


### Training data Embeddings

In [89]:
X=np.zeros((len(sentences),num_words,vec_size))
y=np.zeros((len(sentences),len(vocab)))
for ix in range(len(sentences)):
    
    # y[ix, ] = word_model.wv[next_word[ix]]
    y[ix,word_ix[next_word[ix]]] = 1

    words = sentences[ix].split(" ")
    if ix%50000==0:
        print(ix, len(words))
        print(sentences[ix])
        print(words)
    for iy in range(len(words)):
        X[ix,iy] = word_model.wv[words[iy]]

0 10
ugh ha ha ha i m serious nigga one of
['ugh', 'ha', 'ha', 'ha', 'i', 'm', 'serious', 'nigga', 'one', 'of']
50000 10
the bando whoo trapped out the bando bando trapped out
['the', 'bando', 'whoo', 'trapped', 'out', 'the', 'bando', 'bando', 'trapped', 'out']


In [90]:
print(X.shape, y.shape)

(87784, 10, 300) (87784, 9182)


In [91]:
model=Sequential()
model.add(LSTM(256,input_shape=(num_words,vec_size)))
model.add(Dense(len(vocab)))
model.add(Activation('softmax'))
model.summary()
model.compile(optimizer=Adam(learning_rate=0.0001),loss='categorical_crossentropy')

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_4 (LSTM)               (None, 256)               570368    
                                                                 
 dense_3 (Dense)             (None, 9182)              2359774   
                                                                 
 activation_3 (Activation)   (None, 9182)              0         
                                                                 
Total params: 2,930,142
Trainable params: 2,930,142
Non-trainable params: 0
_________________________________________________________________


In [92]:
model.fit(X,y,epochs=5,batch_size=256)

Epoch 1/5
343/343 [==============================] - 51s 135ms/step - loss: 7.0087
Epoch 2/5
343/343 [==============================] - 50s 147ms/step - loss: 6.5004
Epoch 3/5
343/343 [==============================] - 52s 153ms/step - loss: 6.4857
Epoch 4/5
343/343 [==============================] - 54s 158ms/step - loss: 6.4695
Epoch 5/5
343/343 [==============================] - 58s 169ms/step - loss: 6.4461


In [68]:
#serialize model to JSON  serialize model to JSON
# model_json = model.to_json()
# with open("model.json", "w") as json_file:
#     json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("./../dataset/models/model_word_300.h5")
print("Saved model to disk")

Saved model to disk


In [93]:
# import random
np.random.seed(5)
# generated=''
# start_index=random.randint(0,len(txt)-maxlen-1)
# sent=txt[start_index:start_index+maxlen]

print(train_rap.loc[0]["lyrics"][0:400])
test_str = "(Ugh) Ha-ha-ha, I'm serious, nigga One of"
generated = test_str.lower()
actual_lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  train_rap.loc[0]["lyrics"].lower().strip())
actual_lyric = " ".join([s for s in actual_lyric if s!=''][0:60])

generated = " ".join(split_str(" ;,-'\\\'\"()[].?!:{}",  generated.strip()))

print(actual_lyric)
# generated+=sent
for i in range(50):
    x_sample = generated.split(" ")[i:i+num_words]

    x=np.zeros((1,num_words,vec_size))
    for w in range(num_words):
        x[0, w] = word_model.wv[w]
    pred = model.predict(x)
    pred = np.reshape(pred, pred.shape[1])

    ix=np.random.choice(range(len(vocab)),p=pred.ravel())
    generated+= " " + ix_word[ix]
print(generated)

(Ugh) Ha-ha-ha, I'm serious, nigga One of y'all niggas got some bad motherfuckin' breath (Oh, man) Aye, baby, aye, baby, (shit) aye, baby Get some bubblegum in this motherfucker or somethin' Aye, nigga, get somethin' to eat, dog Aye, nigga, study long, study wrong, nigga  With so much drama in the L-B-C It's kind of hard bein' Snoop D-O-double-G But I, somehow, some way Keep comin' up with funky-a
ugh ha ha ha i m serious nigga one of y all niggas got some bad motherfuckin breath oh man aye baby aye baby shit aye baby get some bubblegum in this motherfucker or somethin aye nigga get somethin to eat dog aye nigga study long study wrong nigga with so much drama in the l b c it s
 ugh  ha ha ha  i m serious  nigga one of yeah it change t in from you we couple can jersey why the got it damn think dpg master i i bitches shout one to said cocaina and and can coming twelve birth but this i down we pop a birds start it t down some a do cause o


In [157]:
# embedd_index

In [ ]:
print(generated)

In [1]:
print(X[0][0].shape)
print(similar_by_vector(X[0][0]))

NameError: name 'X' is not defined